# Train Your First Machine Learning Model

Companion notebook for the course project [Train Your First Machine Learning Model](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/ml-classifier).

This notebook trains and compares two binary classifiers — `LogisticRegression` and `RandomForestClassifier` — on the classic Titanic dataset, predicting whether a passenger survived from features like age, sex, and class.

**No API key, no GPU, no internet beyond one small CSV download.** That makes this project a genuinely good fit for Colab/Kaggle/Binder — training either model on a dataset this small (a few hundred rows at most) takes seconds on a free CPU-only notebook. **Kaggle Notebooks specifically** is a nice full-circle choice: the Titanic dataset is itself one of Kaggle's original, most famous beginner competitions, so running this notebook there means training a model on Kaggle's own platform, on Kaggle's own dataset.

See the [full lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/ml-classifier) for the step-by-step walkthrough and the reasoning behind each step.

## Step 0: Install dependencies

Run this once per notebook session (Colab/Kaggle/Binder all support `!pip install` cells).

In [ ]:
!pip install scikit-learn pandas


## Step 1: Load and prepare the data

Same Titanic dataset used in Data Analysis Week 10's guided EDA, loaded here from the course's raw dataset file.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/static/datasets/titanic.csv"
df = pd.read_csv(url)
df.head()


Quick recap of the cleaning Week 10 already walked through in depth — just enough here to get a clean DataFrame, not re-taught.

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["PassengerId", "Name"])  # identifiers, not predictive signal


### Encoding categorical columns

`Sex` and `Embarked` are strings ("male"/"female", "S"/"C"/"Q") — scikit-learn's models only accept numeric input, so every categorical column has to be converted first. `pd.get_dummies` turns one categorical column into several 0/1 columns, one per category. `drop_first=True` drops one category per column since it's fully implied by the others being 0.

In [ ]:
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
df.head()


Finally, separate the columns you're predicting *from* (the features, `X`) from the column you're predicting (the target, `y`).

In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]


## Step 2: Split into training and test sets

A model's score on data it was trained on tells you almost nothing about how it'll do on data it hasn't seen — a model can simply memorize training rows rather than learn a genuine pattern. Holding back a test set gives an honest measure of performance on passengers the model never trained on.

`test_size=0.2` holds back 20% of the rows for testing. `random_state=42` fixes the random shuffle so the split (and therefore the accuracy numbers below) is reproducible across reruns.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape


## Step 3: Train a classifier

`LogisticRegression`, despite the name, is a classifier: it computes a weighted sum of each passenger's features, then squashes that sum through the logistic/sigmoid function into a probability of survival. "Fitting the model" means finding the weights that make those estimated probabilities line up with the actual 0/1 outcomes in the training data. `.fit()` never sees `X_test`/`y_test` — that's what keeps the later evaluation honest.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
predictions[:10]


## Step 4: Evaluate and compare models

The single number to start with is accuracy — the fraction of test-set predictions that matched the real outcome.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

accuracy = accuracy_score(y_test, predictions)
print(f"Logistic Regression accuracy: {accuracy:.1%}")


Accuracy alone hides *what kind* of mistakes the model makes. A confusion matrix breaks that down: how many passengers who actually died were correctly predicted to die, how many were wrongly predicted to survive, and vice versa.

In [ ]:
cm = confusion_matrix(y_test, predictions)
print(cm)


Now train a second, different kind of model on the exact same split, and compare honestly. A random forest trains many small decision trees on random subsets of the data/features and has them vote on the final prediction — a different underlying idea from logistic regression's single weighted-sum-plus-probability approach.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Random Forest accuracy: {rf_accuracy:.1%}")
print(confusion_matrix(y_test, rf_predictions))


## What you just built

A model that generalizes from examples it saw to a prediction about examples it didn't — the same shape of workflow (prepare data, split honestly, fit, evaluate, compare) used for far more sophisticated models. See the [full lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/ml-classifier) for the pitfalls to watch out for (data leakage, over-interpreting a small accuracy gap on this small a test set) and where to go from here (feature engineering, cross-validation, a different Kaggle dataset).